# Evaluate a Liga board with Lc0 on a Colab A100

1. Choose **Runtime → Change runtime type → GPU → A100** and connect.
2. Run setup cells 1–4. They build the stable **Lc0 v0.32.1** CUDA backend and download **BT4-it332**, recommended by the Leela project for large GPUs. First setup can take several minutes; the network download is about 383 MB.
3. Pause your Liga replay on the position you want, export it using the snippet below, and paste the JSON into cell 5. FEN alone also works, but JSON keeps the recorded move history used by Leela and repetition detection.
4. Run cells 5–7. Start with 120 seconds and three candidate moves; use longer analysis or one candidate for a deeper single-line search.

This is a separate analysis notebook. It does not modify or connect an analysis service to your website. The GPU build must run in Colab; it has not been executed on an A100 by the notebook author.

For quick FEN-only analysis, pause the website replay and press **Ctrl+C** (Cmd+C on macOS), then paste into cell 5. Use the JSON export below to preserve the recorded move history.

## Export the currently displayed position
On your desktop browser, open a game review, pause it, choose a move, then open **F12 → Console** and run this read-only snippet. It copies position data to your clipboard; it does not send data anywhere.

```javascript
(() => {
  const source = document.getElementById('replay-data');
  if (!source) throw new Error('Open a recorded game replay first.');
  if (document.querySelector('[data-action="play"]')?.getAttribute('aria-pressed') === 'true') {
    throw new Error('Pause the replay before exporting.');
  }
  const replay = JSON.parse(source.textContent);
  const buttons = [...document.querySelectorAll('[data-moves] button.replay-move')];
  const ply = buttons.findIndex(button => button.getAttribute('aria-current') === 'step') + 1;
  copy(JSON.stringify({
    fen: replay.positions[ply],
    initial_fen: replay.positions[0],
    moves: replay.moves.slice(0, ply).map(move => move.uci)
  }, null, 2));
})();
```

The full FEN includes side to move, castling and en-passant state. Flipping the visual board does not change the position. The exported JSON preserves the available game history from the PGN's starting position.

## 1. Confirm the A100 and CUDA toolkit

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

subprocess.run(['nvidia-smi'], check=True)
gpu_name = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True)
assert 'A100' in gpu_name, f'Select an A100 runtime first. Current GPU: {gpu_name.strip()}'
nvcc = shutil.which('nvcc') or '/usr/local/cuda/bin/nvcc'
assert Path(nvcc).is_file(), 'CUDA toolkit (nvcc) missing from this runtime.'
subprocess.run([nvcc, '--version'], check=True)
CUDA_ROOT = Path(nvcc).resolve().parent.parent
print('CUDA toolkit:', CUDA_ROOT)

## 2. Install build tools and python-chess
Colab already supplies NVIDIA drivers and CUDA. This cell does not install or replace GPU drivers, PyTorch or cuDNN.

In [ ]:
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'git', 'g++', 'pkg-config', 'zlib1g-dev'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'meson>=1.4,<2', 'ninja', 'python-chess==1.999'], check=True)

## 3. Build Lc0 for the A100
The CUDA FP16 backend uses A100 tensor cores. cuDNN is deliberately unnecessary for this backend. Two build jobs limit compiler memory use.

In [ ]:
SOURCE = Path('/content/lc0-source')
BUILD = SOURCE / 'build' / 'colab'
if not SOURCE.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'v0.32.1', '--recurse-submodules',
                    'https://github.com/LeelaChessZero/lc0.git', str(SOURCE)], check=True)
head = subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip()
tag = subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'v0.32.1^{commit}'], text=True).strip()
assert head == tag, 'The existing source directory is not at v0.32.1.'
env = os.environ.copy()
env['PATH'] = str(CUDA_ROOT / 'bin') + os.pathsep + env['PATH']
libraries = [str(CUDA_ROOT / 'lib64'), str(CUDA_ROOT / 'targets/x86_64-linux/lib')]
env['LD_LIBRARY_PATH'] = os.pathsep.join(libraries) + os.pathsep + env.get('LD_LIBRARY_PATH', '')
options = ['--buildtype=release', '-Dgtest=false', '-Dplain_cuda=true', '-Dcudnn=false',
           '-Dcc_cuda=80', '-Dnative_arch=false', '-Dblas=false', '-Donnx=false', '-Dmetal=disabled',
           '-Dcudnn_include=' + str(CUDA_ROOT / 'include'),
           '-Dcudnn_libdirs=' + ','.join(libraries)]
setup = ['meson', 'setup', str(BUILD), str(SOURCE)]
if (BUILD / 'build.ninja').exists():
    setup.append('--reconfigure')
subprocess.run(setup + options, check=True, env=env)
subprocess.run(['meson', 'compile', '-C', str(BUILD), '-j', '2'], check=True, env=env)
LC0 = BUILD / 'lc0'
assert LC0.is_file(), 'Lc0 binary was not produced.'
subprocess.run([str(LC0), '--version'], check=True, env=env)
print('Engine:', LC0)

## 4. Download the neural network
BT4-it332 is linked on the official [best networks page](https://lczero.org/play/networks/bestnets/). Keep the `.pb.gz` file compressed. Downloads are cached for this runtime; Colab resets remove `/content` files. Save the weight file to Drive manually if you want to reuse it in later sessions.

In [ ]:
import urllib.request, gzip, hashlib
NETWORK_URL = 'https://storage.lczero.org/files/networks-contrib/BT4-1024x15x32h-swa-6147500-policytune-332.pb.gz'
WEIGHTS = Path('/content/lc0-assets/BT4-it332.pb.gz')
WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
if not WEIGHTS.exists():
    partial = WEIGHTS.with_suffix('.download')
    urllib.request.urlretrieve(NETWORK_URL, partial)
    with gzip.open(partial, 'rb') as stream:
        while stream.read(1024 * 1024):
            pass  # Validate the gzip stream before keeping the download.
    partial.replace(WEIGHTS)
with WEIGHTS.open('rb') as stream:
    NETWORK_SHA256 = hashlib.file_digest(stream, 'sha256').hexdigest()
print('Network:', WEIGHTS)
print('SHA-256 (recorded for reproducibility):', NETWORK_SHA256)

## 5. Paste the exported position
Replace the text between the triple quotes below with your JSON or a complete FEN. The SAN variations and WDL percentages in the output will be calculated for that position.

In [ ]:
POSITION = r"""PASTE YOUR JSON OR FEN HERE"""
SECONDS = 120
CANDIDATE_MOVES = 3

## 6. Validate and display the position
Check the board and side to move before spending GPU time.

In [ ]:
import json, chess, chess.engine, chess.svg
from IPython.display import display, SVG

def board_from_export(raw):
    raw = raw.strip()
    if not raw or raw == 'PASTE YOUR JSON OR FEN HERE':
        raise ValueError('Paste your exported JSON or FEN in cell 5 first.')
    if raw.startswith('{'):
        payload = json.loads(raw)
        board = chess.Board(payload['initial_fen'])
        for uci in payload['moves']:
            board.push_uci(uci)
        if board.fen() != chess.Board(payload['fen']).fen():
            raise ValueError('The move history does not reproduce the exported FEN.')
    else:
        board = chess.Board(raw)
    if not board.is_valid():
        raise ValueError('Invalid chess position: check the FEN.')
    return board

board = board_from_export(POSITION)
print('FEN:', board.fen())
print('Side to move:', 'White' if board.turn else 'Black')
print('Recorded history:', len(board.move_stack), 'plies')
display(SVG(chess.svg.board(board=board, size=400)))

## 7. Run GPU analysis
Evaluations and W/D/L are shown from **White's perspective**: positive favours White; negative favours Black. Lc0 centipawns are a converted evaluation, not a count of material. Win/draw/loss is often a better way to interpret its assessment. The PV is a suggested continuation, not a forced sequence.

Use 120 seconds to start, 300–600 seconds for deeper analysis. MultiPV 1 concentrates the search on the best line; MultiPV 3 compares candidates. Search depth and node counts are not directly comparable with Stockfish.

In [ ]:
if not 1 <= int(SECONDS) <= 3600:
    raise ValueError('SECONDS must be between 1 and 3600.')
if not 1 <= int(CANDIDATE_MOVES) <= 10:
    raise ValueError('Choose between 1 and 10 candidate moves.')
if board.is_game_over():
    raise ValueError(f'This is already a finished position: {board.outcome()}. Choose an earlier move.')

stderr_log = Path('/content/lc0-engine.log')
records = []
with stderr_log.open('w') as errors:
    engine = chess.engine.SimpleEngine.popen_uci(
        [str(LC0), '--backend=cuda-fp16', '--weights=' + str(WEIGHTS),
         '--threads=2', '--minibatch-size=64'], timeout=180, stderr=errors, env=env)
    try:
        if 'UCI_ShowWDL' in engine.options:
            engine.configure({'UCI_ShowWDL': True})
        count = min(int(CANDIDATE_MOVES), board.legal_moves.count())
        infos = engine.analyse(board, chess.engine.Limit(time=int(SECONDS)), multipv=count)
        for rank, info in enumerate(infos, 1):
            pv = info.get('pv', [])
            if not pv:
                continue
            white_score = info['score'].pov(chess.WHITE)
            mate = white_score.mate()
            evaluation = f'Mate {mate:+d}' if mate is not None else f'{white_score.score()/100:+.2f}'
            row = {'rank': rank, 'best_move': board.san(pv[0]), 'evaluation_white': evaluation,
                   'pv_san': board.variation_san(pv), 'pv_uci': [m.uci() for m in pv],
                   'nodes': info.get('nodes'), 'depth': info.get('depth')}
            if info.get('wdl') is not None:
                wdl = info['wdl'].pov(chess.WHITE)
                total = sum(wdl)
                row['wdl_white_percent'] = [round(v * 100 / total, 1) for v in wdl]
            records.append(row)
            print(f"{rank}. {row['best_move']} | Evaluation (White): {evaluation}")
            if 'wdl_white_percent' in row:
                print('   White win / draw / Black win (%):', row['wdl_white_percent'])
            print('   Line:', row['pv_san'])
        print('GPU process memory (the CUDA backend is forced, with no CPU fallback):')
        subprocess.run(['nvidia-smi', '--query-compute-apps=process_name,used_memory', '--format=csv'], check=True)
    except Exception:
        print('Engine log:', stderr_log)
        print(stderr_log.read_text()[-5000:])
        raise
    finally:
        engine.quit()

OUTPUT = Path('/content/lc0-analysis.json')
OUTPUT.write_text(json.dumps({'fen': board.fen(), 'initial_fen': board.root().fen(),
    'moves': [m.uci() for m in board.move_stack], 'engine_version': 'v0.32.1',
    'network_url': NETWORK_URL, 'network_sha256': NETWORK_SHA256,
    'seconds': int(SECONDS), 'multipv': count, 'lines': records}, indent=2))
print('Saved:', OUTPUT)

## 8. Optional: download the analysis
Run this cell when you want to save the result to your PC.

In [ ]:
from google.colab import files
files.download(str(OUTPUT))

## Troubleshooting

- **GPU unavailable:** run cell 1 and confirm the assigned device is an A100.
- **No CUDA backend / missing CUDA libraries:** inspect cell 3's Meson summary. `nvcc`, `libcublas` and `libcudart` must come from the runtime's CUDA toolkit. Do not solve this by selecting a CPU backend if you want GPU analysis.
- **CUDA/compiler error:** read the first build error, check `nvcc --version` and `g++ --version`, and use a host compiler supported by that CUDA release. The build option `-Dnvcc_ccbin=/path/to/supported/g++` can select it.
- **Out of memory:** change `--minibatch-size=64` to `--minibatch-size=16`, or use the smaller BT3 network from the official network page.
- **Slow first analysis:** loading the large network and initializing CUDA adds startup time. Subsequent work in the same engine process is faster.
- **Runtime reset:** rerun setup and load your position again. The Colab process does not run continuously after the session ends.

Official references: [Lc0 v0.32.1 source/build instructions](https://github.com/LeelaChessZero/lc0/tree/v0.32.1), [recommended networks](https://lczero.org/play/networks/bestnets/).